In [ ]:
library(DOTr)
library(ggplot2)
library(SeuratObject)
library(Seurat)
library(presto)
library(dplyr)
library(cowplot)
library(SeuratDisk)

In [ ]:
sc <-readRDS('data/spatial/processed_data/scRNA.rds')

In [ ]:
st <-readRDS('data/spatial/processed_data/spatial.rds')
st

In [ ]:
donors = c('BCLL-8-T','BCLL-9-T','BCLL-10-T','BCLL-11-T','BCLL-12-T','BCLL-13-T')
prefix = c('c28w2r_7jne4i_','qvwc8t_2vsr67_','esvq52_nluss5_','exvyh1_66caqq_','p7hv1g_tjgmyj_','gcyl7c_cec61b_')

In [ ]:
table(st@meta.data$donor_id)

In [ ]:
coor <- read.csv("data/spatial/coordinates.csv", header = TRUE, stringsAsFactors = FALSE)

In [ ]:
donor<-'BCLL-8-T'
#samples <- rownames(sc@meta.data)[sc$donor_id == donor&!sc$annotation_level_1 %in% c("preTC", "preBC")]
samples <- rownames(sc@meta.data)[!sc$annotation_level_1 %in% c("preTC", "preBC")]

sc_counts <- GetAssayData(sc, assay = "RNA", slot = "counts")[, samples]
#sc_counts <- sc[["RNA"]]@counts[,samples]

In [ ]:
#label <- droplevels(sc@meta.data[sc$donor_id == donor&!sc$annotation_level_1 %in% c("preTC", "preBC"), ]$annotation_level_1)
label <- droplevels(sc@meta.data[!sc$annotation_level_1 %in% c("preTC", "preBC"), ]$annotation_level_1)
#label <- sc@meta.data$cell_type
#label <- sc@meta.data$annotation_level_1

In [ ]:
samples <- rownames(st@meta.data)[st$donor_id == donor]
st_counts <- GetAssayData(st, assay = "Spatial", slot = "counts")[, samples]
#st_counts <- st_counts[, samples, drop = FALSE]

In [ ]:
slice <- 'c28w2r_7jne4i'
xy <- coor[startsWith(coor$X, slice), ]
xy <- xy[, c("row", "col")]
xy <- as.matrix(xy)
rownames(xy) <- NULL
colnames(xy) <- NULL

In [ ]:
dot.srt <- setup.srt(srt_data = st_counts, srt_coords = xy,th.spatial = 0.8)

In [ ]:
dot.ref <- setup.ref(ref_data = sc_counts, ref_annotations =label , 8, max_genes = 4000, verbose = FALSE)

In [ ]:
dot <- create.DOT(dot.srt, dot.ref)

In [ ]:
dot <- run.DOT.lowresolution(dot,  
                ratios_weight = 0.2,       
                max_spot_size = 20,
                iterations = 100,
                verbose = FALSE)    
dim(dot@weights)

In [ ]:
dim(dot@solution)

In [ ]:
dot_file <- paste0('data/spatial/DOT_output/','BCLL-9-T' , '.rds')
saveRDS(dot, file = dot_file)
#saveRDS(dot, file = 'data/spatial/DOT_output/donor_9.rds')

In [ ]:
#donor<-'BCLL-8-T'
#dot <- readRDS('data/spatial/DOT_output/donor_8.rds')
dot_file <- paste0('data/spatial/DOT_output/', 'BCLL-9-T', '.rds')
dot <- readRDS(dot_file)

In [ ]:
for (i in 1:ncol(dot@weights)) {
  
  plot_data <- data.frame(
    Col = xy[,1],
    Row = xy[,2],
    value = dot@weights[, i]
  )
  

  p <- ggplot(plot_data, aes(x = Col, y = Row, color = value)) +
    geom_point(size = 3) +
    scale_color_gradient(
      low = "lightgrey", 
      high = "red", 
    ) +
    theme_bw() +
    theme(
      panel.background = element_rect(fill = 'white'),
      panel.grid = element_blank(),
      axis.text = element_blank(),
      axis.title = element_blank(),
      axis.ticks = element_blank()
    ) +
    ggtitle(colnames(dot@weights)[i])
  
  print(p)  
  #ggsave(filename = paste0("plot_", colnames(dot@weights)[i], ".png"), plot = p, width = 8, height = 8, dpi = 300)
}

In [ ]:
sp <-readRDS('data/spatial/20220527_tonsil_atlas_spatial_seurat_obj.rds')
sp

In [ ]:
rownames(sp@meta.data) <- Cells(sp)

In [ ]:
colnames(sp@meta.data)

In [ ]:
p <- SpatialDimPlot(
  object = sp,
  images = slice,
  group.by = 'annotation_20220215',
  pt.size.factor = 120
)
print(p)
ggsave("spatial_plot.png", p, width = 8, height = 8, dpi = 300)

In [ ]:
plot_data <- data.frame(xy)
plot_data$celltype <- colnames(dot@weights)[apply(dot@weights, 1, which.max)]
ggplot(plot_data, aes(x =xy[,1] , y = xy[,2], color = celltype))+
  geom_point(size = 3)+
  theme_bw()+
  theme(panel.background = element_rect(fill = 'white'), 
        panel.grid = element_blank(),
        axis.text = element_blank(), 
        axis.title = element_blank(), 
        axis.ticks = element_blank())

In [ ]:
plot_list <- list()
for (i in 1:ncol(dot@weights)) {
  plot_data <- data.frame(
    Col = xy[,2],  
    Row = -xy[,1], 
    value = dot@weights[, i]
  )
  
  p <- ggplot(plot_data, aes(x = Col, y = Row, color = value)) +
    geom_point(size = 2) + 
    scale_color_gradient(low = "lightgrey", high = "red") +
    theme_void() + 
    theme(
      legend.position = "bottom",
      legend.key.height = unit(0.3, "cm"),
      plot.title = element_text(hjust = 0.5, size = 10)
    ) +
    ggtitle(colnames(dot@weights)[i])
  
  plot_list[[i]] <- p
}

combined_plot <- plot_grid(plotlist = plot_list, ncol = 4, align = 'hv')
print(combined_plot)
ggsave("combined_spatial_plots.png", combined_plot, 
       width = 16, height = 12, dpi = 300)